# 4. Aggregation

> Aggregate and combine the four cleaned datasets for downstream analysis and modelling.

## 1. Setup

In [ ]:
import importlib
import sys, os

sys.path.insert(0, os.path.abspath(".."))

import src.code.io_utils as io

importlib.reload(io)

import pandas as pd
import numpy as np

## 2. Load Datasets

In [ ]:
bdoss_clean    = io.load("../data/prepared/bdoss_clean.parquet")
crc_clean      = io.load("../data/prepared/crc_clean.parquet")
credscore_clean = io.load("../data/prepared/credscore_clean.parquet")
fama_clean     = io.load("../data/prepared/fama_clean.parquet")

## 3. Aggregate BDOSS → one row per contract (CONTRIB × DOSSIER)

In [ ]:
print("bdoss_clean:    ", bdoss_clean.shape)
print("crc_clean:      ", crc_clean.shape)
print("credscore_clean:", credscore_clean.shape)
print("fama_clean:     ", fama_clean.shape)

In [ ]:
# Sort so that "last" aggregation = most recent observation per contract
bdoss_clean = bdoss_clean.sort_values(["CONTRIB", "DOSSIER", "OBS_DATE"])

In [ ]:
# Derive binary flags from POS before dropping it
has_san = (
    bdoss_clean.groupby(["CONTRIB", "DOSSIER"])["POS"]
    .apply(lambda s: int((s == "SAN").any()))
    .rename("HAS_SAN")
)
has_rbt = (
    bdoss_clean.groupby(["CONTRIB", "DOSSIER"])["POS"]
    .apply(lambda s: int((s == "RBT").any()))
    .rename("HAS_RBT")
)

In [ ]:
drop_cols = ["OBS_DATE", "POS", "DCSP", "PTT"]
bdoss_agg_input = bdoss_clean.drop(columns=drop_cols)

agg_dict = {
    "DPOS":        "last",
    "DCREAT":      "last",
    "DATFIN":      "last",
    "D1FIN":       "last",
    "DURDEG":      "max",
    "RANGPRO":     "max",
    "RANGCLI":     "max",
    "MTFINO":      "max",
    "MTFIN":       "max",
    "MENSALIDADE": "median",
    "CRD":         "last",
    "SREC":        "last",
    "RN":          "max",
    "RD":          "max",
    "RISK":        "last",
    "RISKA":       "last",
    "RESSO":       "last",
    "CSP":         "last",
    "NBENF":       "last",
}

bdoss_aggregated = (
    bdoss_agg_input
    .groupby(["CONTRIB", "DOSSIER"], sort=False)
    .agg(agg_dict)
    .reset_index()
)

In [ ]:
# Attach HAS_SAN and HAS_RBT
bdoss_aggregated = bdoss_aggregated.merge(
    pd.concat([has_san, has_rbt], axis=1).reset_index(),
    on=["CONTRIB", "DOSSIER"],
    how="left"
)

In [ ]:
print(f"Input rows  : {bdoss_clean.shape[0]:,}")
print(f"Output rows : {bdoss_aggregated.shape[0]:,}")
print(f"Columns     : {bdoss_aggregated.shape[1]}")

nulls = bdoss_aggregated.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\nNull counts:\n{nulls if not nulls.empty else 'None'}")

bdoss_aggregated.head(3)

In [ ]:
os.makedirs("../data/prepared", exist_ok=True)
io.save(bdoss_aggregated, "../data/prepared/bdoss_aggregated.parquet")